# 1 - Signals Generation from Risk-Neutral Density

## Sections
1) Data & Preprocessing

2) Calls Subset and Base Features

3) IV Smile Construction & Interpolation

4) Pricing Model (Black–Scholes)

5) Risk-Neutral Density (Breeden–Litzenberger)

6) Daily Signal Definition & Selection

7) Output Artifacts

8) Closing Notes

---

## 1) Data & Preprocessing

#### <u>Dataset & Standardization</u>

In this section the code:
- Loads the Deribit BTC options dataset. 
- Standardizes column naming (Camel Case).
- Cleans `Option_Type`. 
- Parses dates.
- Deduplicates and sorts by `Date → Expiry_Date → Strike`.

In [ ]:
# ===== SETUP DATA FOR BACKTEST =====
import pandas as pd
from tqdm import tqdm
import numpy as np
from scipy.stats import norm
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

In [ ]:
# Load options dataset
df_all = pd.read_csv("Data/options_ready_for_backtest.csv")

# Function to transform column names to Camel Case (only first letter capitalized)
def to_camel_case(s):
    return '_'.join(word.capitalize() for word in s.lower().split('_'))

# Rename columns to Camel Case format
df_all.columns = [to_camel_case(col) for col in df_all.columns]

# Clean option type values (remove spaces) and standardize
if df_all["Option_Type"].str.contains(" ").any():
    df_all["Option_Type"] = df_all["Option_Type"].str.strip()
df_all["Option_Type"] = df_all["Option_Type"].map({"call": "Call", "put": "Put"})

# Convert dates to datetime format
df_all["Date"] = pd.to_datetime(df_all["Date"])
df_all["Expiry_Date"] = pd.to_datetime(df_all["Expiry_Date"])

# Remove duplicates and sort
df_all = df_all.drop_duplicates().reset_index(drop=True)
df_all = df_all.sort_values(by=["Date", "Expiry_Date", "Strike"]).reset_index(drop=True)

---

## 2) Calls Subset and Base Features

#### <u>Filtering & Base Quantities</u>

- Keep **Calls** only.  
- Drop rows with missing `Mark_Iv`, `Underlying_Price`, `Strike`, `Expiry_Date`, `Date`, `Dte`.  
- Compute time-to-maturity:

$$ T = \text{Dte}/365$$
  
- Compute **moneyness**: 

$$( K/S )$$ 

- And retain:

$$( 0.7 \le K/S \le 1.3 )$$


In [ ]:
# ===== FILTER ATM CALLS ONLY FOR CALCULATIONS =====
df_calls = df_all[df_all["Option_Type"] == "Call"].copy()
df_calls = df_calls.dropna(subset=["Mark_Iv", "Underlying_Price", "Strike", "Expiry_Date", "Date", "Dte"])
df_calls["T"] = df_calls["Dte"] / 365
df_calls["Moneyness"] = df_calls["Strike"] / df_calls["Underlying_Price"]
df_calls = df_calls[(df_calls["Moneyness"] >= 0.7) & (df_calls["Moneyness"] <= 1.3)]

---

## 3) IV Smile Construction & Interpolation

#### <u>Linear Interpolation on a Uniform Strike Grid</u>

For each (Date, Expiry):

- Build the IV smile from observed quotes.  
- Interpolate IV linearly across strikes.  
- Evaluate on a **uniform dense grid** of 200 strikes in $$[\min K, \max K]$$ to stabilize curvature estimates used for RND.

---

## 4) Pricing Model (Black–Scholes)

#### <u>Call Pricing on the Dense Strike Grid</u>

- Given:
$$
Spot (S), \qquad
Strike (K), \qquad
IV (\sigma), \qquad
Maturity (T):
$$

- We apply the **Black–Scholes** model:

$$
d_1 = \frac{\ln(S/K) + \tfrac{1}{2}\sigma^2 T}{\sigma \sqrt{T}}
$$  

$$
d_2 = d_1 - \sigma \sqrt{T}
$$  

$$
C(S,K,T) = S \, \Phi(d_1) - K \, \Phi(d_2)
$$

- Calls are priced on the dense strike grid using the interpolated implied volatility.


In [ ]:
def bs_call(k, s, sigma, t):
    d1 = (np.log(s / k) + 0.5 * sigma**2 * t) / (sigma * np.sqrt(t))
    d2 = d1 - sigma * np.sqrt(t)
    return s * norm.cdf(d1) - k * norm.cdf(d2)

---

## 5) Risk-Neutral Density (Breeden–Litzenberger)

#### <u>Numerical Second Derivative of Call Prices</u>

Under standard assumptions, the risk-neutral density is:

$$
f_Q(K)=e^{rT}\,\frac{\partial^2 C}{\partial K^2}.
$$

Implementation:

- Compute call prices on the dense grid.  
- Approximate $$(\partial^2 C/\partial K^2)$$ via consecutive gradients.  
- **Clip at 0** to enforce non-negativity.  
- Compute **Fair Value** and **Gap**:
$$
\text{Fair Value}=\frac{\sum_i K_i f_Q(K_i)}{\sum_i f_Q(K_i)}, \qquad
\text{Gap}=\frac{\text{Fair Value}-S}{S}.
$$

In [ ]:
# ===== WALK-FORWARD RND EXTRACTION =====
unique_days = sorted(df_calls["Date"].unique())
signals = []

for i in tqdm(range(len(unique_days))):
    today = unique_days[i]
    df_today = df_calls[df_calls["Date"] == today].copy()
    if df_today.empty:
        continue

    expiries = df_today["Expiry_Date"].unique()
    surface = []
    for expiry in expiries:
        df_exp = df_today[df_today["Expiry_Date"] == expiry].copy()
        if len(df_exp) < 5:
            continue
        surface.append({
            "Expiry": expiry,
            "Date": today,
            "Strike": df_exp["Strike"].values,
            "Iv": df_exp["Mark_Iv"].values,
            "Spot": df_exp["Underlying_Price"].iloc[0],
            "T": df_exp["T"].iloc[0]
        })
    if not surface:
        continue

    df_rnd_surface = []
    for surf in surface:
        K = surf["Strike"]
        iv = surf["Iv"]
        S = surf["Spot"]
        T = surf["T"]
        expiry = surf["Expiry"]
        date = surf["Date"]
        if len(K) < 5 or np.any(iv <= 0):
            continue
        try:
            iv_interp = interp1d(K, iv, kind='linear', fill_value='extrapolate')
        except Exception:
            continue
        K_dense = np.linspace(min(K), max(K), 200)
        iv_dense = iv_interp(K_dense) / 100
        def bs_call(k, s, sigma, t):
            d1 = (np.log(s / k) + 0.5 * sigma**2 * t) / (sigma * np.sqrt(t))
            d2 = d1 - sigma * np.sqrt(t)
            return s * norm.cdf(d1) - k * norm.cdf(d2)
        prices = np.array([bs_call(k, S, sigma, T) for k, sigma in zip(K_dense, iv_dense)])
        try:
            d2 = np.gradient(np.gradient(prices, K_dense), K_dense)
        except Exception:
            continue
        density = d2.clip(min=0)
        for k, d in zip(K_dense, density):
            df_rnd_surface.append({
                "Date": date,
                "Expiry": expiry,
                "Strike": k,
                "Density": d,
                "Spot": S,
                "T": T
            })
    df_rnd_surface = pd.DataFrame(df_rnd_surface)
    if df_rnd_surface.empty:
        continue

    df_rnd_summary = (
        df_rnd_surface[["Date", "Expiry", "Strike", "Density"]]
        .assign(weighted_strike=lambda x: x["Strike"] * x["Density"])
        .groupby(["Date", "Expiry"], as_index=False)
        .agg(Fair_Value=("weighted_strike", "sum"), Total_Weight=("Density", "sum"))
    )
    df_rnd_summary["Fair_Value"] = df_rnd_summary["Fair_Value"] / df_rnd_summary["Total_Weight"]
    df_rnd_summary.drop(columns="Total_Weight", inplace=True)
    spot = df_today["Underlying_Price"].dropna().iloc[0]
    df_rnd_summary["Expiry"] = pd.to_datetime(df_rnd_summary["Expiry"]).dt.normalize()
    df_rnd_summary["Gap"] = (df_rnd_summary["Fair_Value"] - spot) / spot

    idx_best = df_rnd_summary["Gap"].abs().idxmax()
    best_expiry_row = df_rnd_summary.loc[idx_best]
    best_expiry = best_expiry_row["Expiry"]
    fair_value = best_expiry_row["Fair_Value"]
    gap = best_expiry_row["Gap"]

    signals.append({
        "Date": today,
        "Spot": spot,
        "Fair_Value": fair_value,
        "Gap": gap,
        "Expiry": best_expiry
    })

---

## 6) Daily Signal Definition & Selection

#### <u>Choose Expiry with Maximum |Gap|</u>

For each **Date**:

- Compute `Fair_Value` and `Gap` across valid `Expiry`.  
- Select the expiry with the **largest absolute Gap**.  
- Retain: `Date`, `Spot`, `Fair_Value`, `Gap`, `Expiry`.

---

## 7) Output Artifacts

#### <u>Interface to Downstream Backtests</u>

We persist **`Data/df_signals.csv`** with:

- `Date` (UTC)  
- `Spot`  
- `Fair_Value`  
- `Gap`  
- `Expiry` (normalized date)

In [ ]:
# Convert signals to DataFrame
df_signals = pd.DataFrame(signals)
df_signals["Date"] = pd.to_datetime(df_signals["Date"])
df_signals["Expiry"] = pd.to_datetime(df_signals["Expiry"])
df_signals.to_csv("Data/df_signals1.csv", index=False)
print("Precomputed signals saved in the Data folder as df_signals1.csv.")

---

## 8) Closing Notes

#### <u>Reproducibility & Caveats</u>

- Deterministic transforms (fixed grid; linear interpolation).  
- Require **≥ 5** strikes per smile; enforce IV > 0.  
- Rates/dividends omitted (can be integrated).  
- Linear interpolation does **not** enforce no-arbitrage; consider SVI/SABR for production.